<a href="https://colab.research.google.com/github/Muffalo52/SDXL-Anima-Colab-trainer/blob/dev/SDXL_Anima_Trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌟 Original : [XL Lora Trainer by Hollowstrawberry](https://colab.research.google.com/github/hollowstrawberry/kohya-colab/blob/main/Lora_Trainer_XL.ipynb)


This Trainer depends on the [LoRA_Easy_Training_scripts_Backend](https://github.com/67372a/LoRA_Easy_Training_scripts_Backend) and [LyCORIS](https://github.com/67372a/LyCORIS) of 67372a

### ⭕ Disclaimer
The purpose of this document is to research bleeding-edge technologies in the field of machine learning.
Please read and follow the [Google Colab guidelines](https://research.google.com/colaboratory/faq.html) and its [Terms of Service](https://research.google.com/colaboratory/tos_v3.html).

In [ ]:
import os
import re
import toml
from time import time
from IPython.display import Markdown, display
import warnings
import subprocess
import shutil

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# os.environ["BNB_CUDA_VERSION"] = "128"

# These carry information from past executions
if "model_url" in globals():
    old_model_url = model_url
else:
    old_model_url = None
if "dependencies_installed" not in globals():
    dependencies_installed = False
if "model_file" not in globals():
    model_file = None

# These may be set by other cells, some are legacy
if "custom_dataset" not in globals():
    custom_dataset = None
if "override_dataset_config_file" not in globals():
    override_dataset_config_file = None
if "continue_from_lora" not in globals():
    continue_from_lora = ""
if "override_config_file" not in globals():
    override_config_file = None

# COMMIT = "d16ecce61d4172d5cee443fc9fb6515fa1bc9275"
COMMIT = "HEAD"
LOAD_TRUNCATED_IMAGES = True
BETTER_EPOCH_NAMES = True
FIX_DIFFUSERS = True

#@title ## 🚩 시작하기

#@markdown ### ▶️ 설정 (Setup)
#@markdown 프로젝트 이름은 이미지가 들어있는 폴더(또는 zip 파일)의 이름과 동일하게 설정하십시오. 공백은 사용할 수 없습니다.
project_name = "" #@param {type:"string"}
project_name = project_name.strip()
#@markdown 폴더 구조는 훈련에 영향을 주지 않으며 순전히 편의를 위한 것입니다. 항상 동일한 구조를 선택하도록 주의해 주십시오.
folder_structure = "Organize by category (MyDrive/lora_training/datasets/project_name)" #@param ["Organize by category (MyDrive/lora_training/datasets/project_name)", "Organize by project (MyDrive/Loras/project_name/dataset)"]
#@markdown 다운로드하여 훈련에 사용할 모델을 결정합니다. 다운로드 링크를 붙여넣거나, `/content/drive/MyDrive`로 시작하는 구글 드라이브 내의 파일 경로를 직접 입력하여 선택할 수도 있습니다.
training_model = "Anima v1.0" #@param ["Illustrious XL 0.1", "Illustrious XL v1.0", "Illustrious XL v2.0", "NoobAI V-Pred 1.0", "NoobAI Eps 1.1", "Stable Diffusion XL 1.0 base", "ChenkinNoob XL v0.2", "ChenkinNoob XL v0.3 Rectified Flow", "Anima Preview", "Anima Preview 2", "Anima Preview 3", "Anima v1.0", "Mugen", "Custom"]
optional_custom_training_model = "" #@param {type:"string"}
#@markdown
custom_model_is_diffusers = False #@param {type:"boolean"}
custom_model_is_vpred = False #@param {type:"boolean"}
custom_model_is_anima = False #@param {type:"boolean"}
#@markdown 시간이 지남에 따른 훈련 진행 상황을 시각화하여 확인하고 싶다면 wandb를 사용해 주십시오.
wandb_key = "" #@param {type:"string"}

#@markdown ### 🔐 API 키 (선택 사항)
#@markdown 비공개 Hugging Face 저장소나 Civitai에서 모델을 다운로드하려면 API 키를 입력해 주십시오.
HF_TOKEN = "" #@param {type:"string"}
CIVITAI_TOKEN = "" #@param {type:"string"}

load_diffusers = custom_model_is_diffusers and len(optional_custom_training_model) > 0
vpred = custom_model_is_vpred and len(optional_custom_training_model) > 0
is_rectified_flow = False

is_anima = (custom_model_is_anima and len(optional_custom_training_model) > 0) or ("Anima" in training_model)

if is_anima:
    # VAE와 Qwen3는 커스텀 모델이든 아니든 항상 동일한 경로 사용
    vae_file = "/content/models/vae/split_files/vae/qwen_image_vae.safetensors"
    qwen3_file = "/content/models/qwen3/split_files/text_encoders/qwen_3_06b_base.safetensors"

    # 커스텀 모델이 아닐 때만 공식 파일 경로 지정
    if not optional_custom_training_model:
        model_url = ""
        if "v1.0" in training_model:
            model_file = "/content/models/anima/split_files/diffusion_models/anima-base-v1.0.safetensors"
        elif "Preview 3" in training_model:
            model_file = "/content/models/anima/split_files/diffusion_models/anima-preview3-base.safetensors"
        elif "Preview 2" in training_model:
            model_file = "/content/models/anima/split_files/diffusion_models/anima-preview2.safetensors"
        else:
            model_file = "/content/models/anima/split_files/diffusion_models/anima-preview.safetensors"
else:
    qwen3_file = None
    vae_file = None

if optional_custom_training_model:
    model_url = optional_custom_training_model
elif "Illustrious" in training_model:
    if "v2.0" in training_model:
        model_url = "https://huggingface.co/OnomaAIResearch/Illustrious-XL-v2.0/blob/main/Illustrious-XL-v2.0.safetensors"
    elif "v1.0" in training_model:
        model_url = "https://huggingface.co/OnomaAIResearch/Illustrious-XL-v1.0/resolve/main/Illustrious-XL-v1.0.safetensors"
    else:
        if load_diffusers:
            model_url = "https://huggingface.co/OnomaAIResearch/Illustrious-xl-early-release-v0"
        else:
            model_url = "https://huggingface.co/OnomaAIResearch/Illustrious-xl-early-release-v0/resolve/main/Illustrious-XL-v0.1.safetensors"
elif "NoobAI Eps" in training_model:
    if load_diffusers:
        model_url = "https://huggingface.co/Laxhar/noobai-XL-1.1"
    else:
        model_url = "https://huggingface.co/Laxhar/noobai-XL-1.1/resolve/main/NoobAI-XL-v1.1.safetensors"
elif "NoobAI V-Pred" in training_model:
    vpred = True
    if load_diffusers:
        model_url = "https://huggingface.co/Laxhar/noobai-XL-Vpred-1.0"
    else:
        model_url = "https://huggingface.co/Laxhar/noobai-XL-Vpred-1.0/resolve/main/NoobAI-XL-Vpred-v1.0.safetensors"
elif "ChenkinNoob XL v0.2" in training_model:
    if load_diffusers:
        model_url = "https://huggingface.co/ChenkinNoob/ChenkinNoob-XL-V0.2"
    else:
        model_url = "https://huggingface.co/ChenkinNoob/ChenkinNoob-XL-V0.2/resolve/main/ChenkinNoob-XL-V0.2.safetensors"
elif "v0.3 Rectified Flow" in training_model:
    is_rectified_flow = True
    if load_diffusers:
        model_url = "https://huggingface.co/ChenkinRF/ChenkinNoob-XL-v0.3-Rectified-Flow"
    else:
        model_url = "https://huggingface.co/ChenkinRF/ChenkinNoob-XL-v0.3-Rectified-Flow/resolve/main/ChenkinNoob-XL-v0.3-Rectified-Flow.safetensors"
elif "Mugen" in training_model:
    is_rectified_flow = True
    model_url = "https://huggingface.co/CabalResearch/Mugen/resolve/main/Mugen.safetensors"
elif is_anima:
    model_url = "" # Anima 모델 및 컴포넌트 다운로드는 별도 분기에서 처리됩니다.
else:
    if load_diffusers:
        model_url = "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0"
    else:
        model_url = "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors"

model_url = model_url.strip()

#@markdown ### ▶️ 데이터 전처리 (Processing)
resolution = 1024 #@param {type:"slider", min:512, max:1536, step:256}
#@markdown 자동 태깅 시 캡션 맨 앞에 추가할 트리거 워드를 지정합니다. 공백으로 두면 추가하지 않습니다.
custom_trigger_word = "" #@param {type:"string"}
custom_trigger_word = custom_trigger_word.strip()
caption_extension = ".txt"
#@markdown 데이터셋 이미지에 좌우 반전 증강(Horizontal flipping augmentation)을 적용합니다.
flip_aug = False #@param {type:"boolean"}
#@markdown 캡션 텍스트 내의 애니메이션 태그 순서를 무작위로 섞으면(Shuffle) 모델의 일반화 및 프롬프트 정확도가 향상됩니다. 단, 활성화 태그(Activation tag)는 항상 파일의 맨 앞에 유지되며 섞이지 않습니다.<p>
shuffle_tags = False #@param {type:"boolean"}
shuffle_caption = shuffle_tags
activation_tags = "1" #@param [0,1,2,3]
keep_tokens = int(activation_tags)
#@markdown ### ▶️ 드롭아웃 설정 (Dropout Settings)
#@markdown 전체 캡션을 무작위로 누락시킬 확률입니다. (스타일 학습 권장: 0.1 ~ 0.15)
caption_dropout_rate = 0 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown 개별 태그(단어)를 무작위로 누락시킬 확률입니다. (스타일 학습 권장: 0.1)
caption_tag_dropout_rate = 0 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#에포크마다 강제로 전체 캡션을 누락시킵니다. (위의 무작위 누락을 기본적으로 사용하므로 0을 권장합니다)
caption_dropout_every_n_epochs = 0
#@markdown ### ▶️ 훈련 스텝 (Steps) <p>
#@markdown 훈련 중 각 이미지가 반복되는 횟수입니다. '총 이미지 수 × 반복 횟수'의 결과가 100 정도가 되도록 설정하거나, 이미지가 100장 이상인 경우 반복 횟수를 1로 설정하는 것을 권장합니다.
num_repeats = 1 #@param {type:"number"}
#@markdown 훈련을 얼마나 오래 진행할지 선택합니다.<p>
#@markdown 1 에포크는 (총 이미지 수 × 반복 횟수) ÷ 배치 크기(Batch size)로 계산된 스텝 수와 동일합니다. <p>
preferred_unit = "Epochs" #@param ["Epochs", "Steps"]
how_many = 80 #@param {type:"number"}
max_train_epochs = how_many if preferred_unit == "Epochs" else None
max_train_steps = how_many if preferred_unit == "Steps" else None
#@markdown 에포크를 더 자주 저장할수록 훈련 진행 과정에 따른 LoRA의 변화를 면밀하게 비교할 수 있습니다.
save_every_n_epochs = 2 #@param {type:"number"}
keep_only_last_n_epochs = 80 #@param {type:"number"}
if not save_every_n_epochs:
    save_every_n_epochs = max_train_epochs
if not keep_only_last_n_epochs:
    keep_only_last_n_epochs = max_train_epochs

#@markdown ### ▶️ 학습 (Learning)
#@markdown 학습률(Learning rate)은 결과 품질에 가장 큰 영향을 미칩니다. 생성된 이미지에 검은색 노이즈나 깨짐 현상이 발생한다면, U-Net과 텍스트 인코더의 학습률을 각각 1e-4와 1e-5 이하로 낮추어 보십시오. <p>
#@markdown Prodigy 계열 옵티마이저 사용 시 lr은 1로 고정합니다.
unet_lr = 0.5 #@param {type:"number"}
text_encoder_lr = 0.5 #@param {type:"number"}
#@markdown 아래 옵션을 활성화하면 텍스트 인코더 훈련이 비활성화되며, `shuffle_tags`, `caption_tag_dropout` 기능 역시 강제로 꺼집니다.
cache_text_encoder_outputs  = True  # @param {type:"boolean"}
#@markdown 스케줄러(Scheduler)는 훈련 진행에 따라 학습률을 어떻게 변화시킬지 결정하는 알고리즘입니다.
lr_scheduler = "constant" #@param ["constant", "cosine", "cosine_with_restarts", "constant_with_warmup", "linear", "polynomial"]
lr_scheduler_number = 1 #@param {type:"number"}
#@markdown 훈련 초반에 학습률을 서서히 끌어올리는 "워밍업(Warm-up)" 구간의 비율입니다. 5% (0.05)로 유지하는 것을 권장합니다.
lr_warmup_ratio = 0.05 #@param {type:"slider", min:0.0, max:0.2, step:0.01}
lr_warmup_steps = 0
#@markdown `min_snr_gamma`는 학습 후반부의 손실(Loss) 수렴을 안정화합니다.
min_snr_gamma_enabled = False #@param {type:"boolean"}
min_snr_gamma = 8 #@param {type:"slider", min:4, max:16.0, step:0.5}
#@markdown `ip_noise_gamma`는 초기 잠재 노이즈를 미세 조정합니다.
ip_noise_gamma_enabled = True #@param {type:"boolean"}
ip_noise_gamma = 0.05 #@param {type:"slider", min:0.05, max:0.1, step:0.01}
#@markdown `noise_offset`은 노이즈 예측 목표값을 미세하게 이동시켜 전체적인 명암비(Contrast)를 개선하는 데 도움을 줍니다.
noise_offset_enabled = False #@param {type:"boolean"}
noise_offset_value = 0.0357 #@param {type:"number"}
#@markdown `Multinoise` (Pyramid Noise Iterations) 옵션은 색상 균형(어두운 곳은 더 어둡게, 밝은 곳은 더 밝게 표현)을 맞추는 데 유용할 수 있습니다.
multinoise = False #@param {type:"boolean"}
#@markdown `flow_shift`는 Rectified Flow 훈련 시 타임스텝 샘플링 분포를 조정합니다. 활성화 시 `uniform` 분포가 적용되며, 비활성화 시 기본 `logit_normal` 분포가 유지됩니다. (SDXL RF 전용)
flow_shift_enabled = False #@param {type:"boolean"}
flow_shift_value = 2.0 #@param {type:"number"}
#@markdown `sigmoid_scale`은 시그모이드(logit_normal) 타임스텝 분포의 분산을 조절합니다. 기본값은 1이며 높을수록 분산이 커집니다. (Anima 전용)
sigmoid_scale = 1.5 #@param {type:"number"}
#@markdown `discrete_flow_shift` 옵션을 활성화하면 타임스텝 샘플링이 `shift`로 변경되며, 노이즈 스케줄을 높은 노이즈 쪽으로 이동시킵니다. (Anima 전용)
discrete_flow_shift_enabled = True #@param {type:"boolean"}
discrete_flow_shift = 2.0 #@param {type:"number"}
#@markdown Anima Flow Matching 최적화 기능 (실험적)
flow_use_ot = True #@param {type:"boolean"}
contrastive_flow_matching = False #@param {type:"boolean"}
#@markdown Focal Frequency Loss (주파수 대역 보조 손실) 옵션을 활성화합니다.
focal_frequency_loss = True #@param {type:"boolean"}
focal_frequency_loss_weight = 1.0 #@param {type:"number"}
focal_frequency_loss_alpha = 1.0 #@param {type:"number"}

#@markdown Latent Wavelet Diffusion (웨이블릿 기반 공간 마스킹) 옵션을 활성화합니다.
wavelet_masking = True #@param {type:"boolean"}
wavelet_mask_l_bound = 0.3 #@param {type:"number"}

#@markdown ### ▶️ 하드웨어 및 성능 (Hardware & Performance)
#@markdown T4 GPU(Colab 무료 티어)를 사용 중이시라면 반드시 이 항목을 체크해 주십시오.
lowram = True #@param {type:"boolean"}
#@markdown (VRAM 사용량을 크게 줄이는 대신 훈련 속도가 약간 감소합니다). G4 이상의 GPU를 사용 중이며 최고 속도를 원하신다면 체크를 해제해 주십시오.
gradient_checkpointing = True #@param {type:"boolean"}
#@markdown 기타 실험적 최적화 기능들입니다.
unsloth_offload_checkpointing = False #@param {type:"boolean"}
attention_mode = "torch" #@param ["torch", "flash", "xformers", "sageattn"]
torch_compile = False #@param {type:"boolean"}
dynamo_backend = "inductor" #@param ["inductor", "eager", "aot_eager", "cudagraphs", "onnxrt", "ipex"]
#@markdown 설치된 파일 폴더를 구글 드라이브에 저장합니다. 약 20분간의 캐시 저장 과정을 거치며 다음 훈련 실행 시 초기 셋업 속도를 5분 이상 단축가능합니다.
cache_installation = False #@param {type:"boolean"}
#@markdown ### ▶️ 아키텍처 구조 (Structure)
#@markdown LoRA는 가장 보편적이고 다양한 목적에 적합한 표준 아키텍처입니다. LoCon과 LoKr 등은 데이터셋의 질감이나 화풍 등 더 복잡한 특징을 학습할 수 있도록 추가 레이어를 활용하기 때문에 스타일 학습에 특히 유리합니다.
lora_type = "LoRA" #@param ["LoRA", "LoCon", "LoHa", "LoKr", "DoKr", "GoRA", "LoRA2", "PiSSA"]

#@markdown dim 값이 클수록 생성되는 LoRA 파일의 용량이 커지며 더 많은 정보를 담을 수 있습니다. 그러나 너무 높은 값이 항상 더 좋은 결과를 보장하는 것은 아닙니다.
network_dim = 32 #@param {type:"number"}
network_alpha = 32 #@param {type:"number"}
#@markdown convolution layer 설정은 SDXL LyCORIS 아키텍처에만 적용됩니다.
conv_dim = 16 #@param {type:"number"}
conv_alpha = 16 #@param {type:"number"}

#@markdown `lokr_factor` 값은 LoKr 아키텍처 전용 매개변수입니다. -1로 설정 시 용량(rank)이 최소화되며 양수값의 경우 0에 가까워질 수록 용량이 커집니다. 기본값 : 8
lokr_factor = 8 #@param {type:"number"}

#@markdown ### ▶️ GoRA 전용 설정 (GoRA 선택 시에만 적용)
#@markdown GoRA는 사전 연산된 그래디언트를 기반으로 레이어마다 적응형 랭크를 할당합니다.
gora_ref_rank = 32 #@param {type:"number"}
gora_min_rank = 16 #@param {type:"number"}
gora_max_rank = 128 #@param {type:"number"}
gora_gamma = 0.05 #@param {type:"number"}

network_module = "networks.lora_anima" if is_anima else "networks.lora"
network_args = None
if lora_type.lower() == "locon":
    # kohya-ss에 내장된 LoCon(C3Lier)을 사용합니다.
    network_args = [f"conv_dim={conv_dim}", f"conv_alpha={conv_alpha}"]

elif lora_type.lower() == "loha":
    # lycoris-lora 패키지를 사용합니다.
    network_module = "lycoris.kohya"
    network_args = [
        f"conv_dim={conv_dim}",
        f"conv_alpha={conv_alpha}",
        "algo=loha"
    ]

elif lora_type.lower() == "lokr":
    # lycoris-lora 패키지를 사용합니다.
    network_module = "lycoris.kohya"
    network_args = [
        f"conv_dim={conv_dim}",
        f"conv_alpha={conv_alpha}",
        "algo=lokr",
    ]
    if is_anima:
        network_args.extend([
            "preset=full",
            "include_patterns=['.*_modulation.*', '.*_embedder.*', '.*final_layer.*']"
        ])

    # lokr_factor 값을 추가합니다
    network_args.append(f"factor={lokr_factor}")

elif lora_type.lower() == "dokr":
    # lycoris-lora 패키지를 사용합니다. (Lokr + Dora)
    network_module = "lycoris.kohya"
    network_args = [
        f"conv_dim={conv_dim}",
        f"conv_alpha={conv_alpha}",
        "algo=lokr",
        "dora_wd=True",
        "wd_on_output=True",
        "decompose_both=False",
        "use_tucker=False",
        "use_scalar=True",
        "train_norm=False",
    ]

    if is_anima:
        network_args.extend([
            "preset=full",
            "include_patterns=['.*_modulation.*', '.*_embedder.*', '.*final_layer.*']"
        ])

    # lokr_factor 값을 추가합니다
    network_args.append(f"factor={lokr_factor}")

elif lora_type.lower() == "gora":
    # lycoris-lora 패키지를 사용합니다.
    network_module = "lycoris.kohya"
    network_args = [
        f"conv_dim={conv_dim}",
        f"conv_alpha={conv_alpha}",
        "algo=gora",
        f"gora_ref_rank={gora_ref_rank}",
        f"gora_min_rank={gora_min_rank}",
        f"gora_max_rank={gora_max_rank}",
        f"gora_gamma={gora_gamma}",
        "gora_importance_type=union_mean",
        "gora_adaptive_gamma=True",
    ]

    if is_anima:
        network_args.extend([
            "preset=full",
            "include_patterns=['.*_modulation.*', '.*_embedder.*', '.*final_layer.*']"
        ])

elif lora_type.lower() == "lora2":
    # lycoris-lora 패키지를 사용합니다.
    network_module = "lycoris.kohya"
    network_args = [
        f"conv_dim={conv_dim}",
        f"conv_alpha={conv_alpha}",
        "algo=lora2",
        "lora2_quantile=0.9",
        "lora2_lambda_r=1e-4",
    ]

    if is_anima:
        network_args.extend([
            "preset=full",
            "include_patterns=['.*_modulation.*', '.*_embedder.*', '.*final_layer.*']"
        ])

elif lora_type.lower() == "pissa":
    # lycoris-lora 패키지를 사용합니다.
    network_module = "lycoris.kohya"
    network_args = [
        f"conv_dim={conv_dim}",
        f"conv_alpha={conv_alpha}",
        "algo=locon",
        "svd_segment=top",
        "pissa_niter=0",
    ]

    if is_anima:
        network_args.extend([
            "preset=full",
            "include_patterns=['.*_modulation.*', '.*_embedder.*', '.*final_layer.*']"
        ])

# network_args가 없을 경우를 대비해 빈 리스트로 초기화 (일반 LoRA 등)
if network_args is None:
    network_args = []

#@markdown ### ▶️ 훈련 세부 설정 (Training)
#@markdown 사용 중인 Colab 환경에 따라 아래 매개변수들을 적절히 조정해 주십시오.
#@markdown
#@markdown 배치 크기(Batch size)를 높이면 훈련 속도는 증가하지만 VRAM 요구량 역시 크게 늘어납니다. T4 최대 Batch Size = 6
train_batch_size = 2 #@param {type:"slider", min:1, max:32, step:1}
#@markdown 지정된 N 스텝 동안 그래디언트를 누적(Gradient Accumulation)하여 업데이트합니다. 이는 메모리를 절약하면서 실질적으로 더 큰 배치 크기를 사용하는 것과 동일한 효과를 냅니다.
gradient_accumulation_steps = 2 #@param {type:"slider", min:1, max:32, step:1}
#@markdown Colab 무료 GPU 티어(T4)에서는 `bf16` 옵션들이 정상적으로 동작하지 않을 수 있습니다. Prodigy 계열은 `full fp16`에서 작동하지 않으니 유의해주세요. <p> `mixed fp16` + SDXL 학습 시 캐싱중 OOM 방지를 위해 2단계 실행과정이 진행됩니다.
precision = "mixed fp16" #@param ["float", "full fp16", "full bf16", "mixed fp16", "mixed bf16"]
#@markdown 최종 저장되는 모델 파일의 정밀도입니다.
save_precision = "fp16" #@param ["fp16", "bf16", "float"]
cache_latents = True
cache_latents_to_drive = True

mixed_precision = "no"
if "fp16" in precision:
  mixed_precision = "fp16"
elif "bf16" in precision:
  mixed_precision = "bf16"
full_precision = "full" in precision

#@markdown ### ▶️ 고급 옵티마이저 설정 (Advanced)
#@markdown 옵티마이저(Optimizer)는 가중치 업데이트에 사용되는 핵심 알고리즘입니다. 기본값인 `AdamW8bit`는 안정적이고 뛰어난 성능을 보입니다. 반면 `Prodigy` 계열은 복잡한 하이퍼파라미터 조율 없이 학습률을 동적으로 자동 관리하므로, 비교적 적은 스텝 수로도 빠르게 수렴하며 특히 소규모 데이터셋 환경에서 두드러진 장점을 제공합니다.
optimizer = "AdamWScheduleFreePlus" #@param ["AdamW8bit", "AdamW8bitKahan", "AdamWScheduleFreePlus", "Prodigy", "ProdigyPlusScheduleFree", "DAdaptation", "DadaptAdam", "DadaptLion", "AdamW", "Lion", "SGDNesterov", "SGDNesterov8bit", "AdaFactor", "Came"]
#@markdown ---
#@markdown ### 💎 Prodigy 옵티마이저 설정
#@markdown `d_coef`는 실질적인 학습률 계산을 전반적으로 제어합니다. (권장 기본값: 1.0)
prodigy_d_coef = 1.5 #@param {type:"number"}
#@markdown 초기 학습률 추정치(d0)입니다. 기본값은 1e-6이며, 학습의 초기 가속을 원하신다면 1e-5 혹은 1e-4 값을 테스트해 보시는 것을 권장합니다.
prodigy_d0 = 1e-6 #@param {type:"number"}
#@markdown ---
#@markdown ### 💎 ProdigyPlusScheduleFree 전용 설정 <p>
#@markdown (※실험적) SPEED 모드를 사용합니다. (메모리 사용량이 최적화되며, 다중 네트워크 훈련 시 효율이 증가합니다)
prodigy_use_speed = False #@param {type:"boolean"}
#@markdown 그래디언트 클리핑을 비활성화(`max_grad_norm=0`)합니다. (Schedule-Free 알고리즘과 충돌을 막기 위해 켜두는 것을 권장합니다)
prodigy_disable_grad_clip = True #@param {type:"boolean"}
#@markdown `split_groups`를 활성화하여 파라미터 그룹별로 독립적인 d 값을 계산합니다. (다중 그룹 학습률 로깅에 최적화됩니다)
prodigy_split_groups = False #@param {type:"boolean"}
#@markdown `split_groups_mean`을 활성화하여 도출된 그룹 간의 d 값을 평균화하여 적용합니다.
prodigy_split_groups_mean = False #@param {type:"boolean"}
#@markdown Schedule-Free 알고리즘의 평균화(Averaging) 강도를 설정합니다. 높을수록 lr 감쇠율이 둔화됩니다. (0으로 설정 시 기본값(20)으로 적용됩니다)
prodigy_schedulefree_c = 200 #@param {type:"number"}

if optimizer == "Prodigy":
    optimizer_args = ["decouple=True", "weight_decay=0.01", "betas=[0.9,0.999]", f"d_coef={prodigy_d_coef}", f"d0={prodigy_d0}", "use_bias_correction=True", "safeguard_warmup=False"]
elif optimizer == "ProdigyPlusScheduleFree":
    optimizer_args = ["betas=[0.95,0.99]", f"d_coef={prodigy_d_coef}", f"d0={prodigy_d0}", "eps=None"]
    if prodigy_use_speed:
        optimizer_args = ["betas=[0.95,0.99]", f"d_coef={prodigy_d_coef}", f"d0={prodigy_d0}", "use_speed=True", "use_orthograd=True"] # SPEED 켜면 decay 제거
    if prodigy_schedulefree_c > 0:
        optimizer_args.append(f"schedulefree_c={prodigy_schedulefree_c}")
    optimizer_args.append(f"split_groups={prodigy_split_groups}")
    optimizer_args.append(f"split_groups_mean={prodigy_split_groups_mean}")
elif optimizer == "AdamW8bit":
    optimizer_args = ["weight_decay=0.1", "betas=[0.9,0.999]"]
elif optimizer == "AdamW8bitKahan":
    optimizer_args = ["weight_decay=0.01", "betas=[0.9,0.99]", "stabilize=False"]
elif optimizer == "AdamWScheduleFreePlus":
    optimizer_args = ["weight_decay=10"]
elif optimizer == "AdaFactor":
    optimizer_args = ["scale_parameter=False", "relative_step=False", "warmup_init=False"]
elif optimizer == "Came":
    optimizer_args = ["weight_decay=0.04"]

if optimizer == "Came":
  optimizer = "LoraEasyCustomOptimizer.came.CAME"
elif optimizer == "AdamW8bitKahan":
  optimizer = "LoraEasyCustomOptimizer.adam.AdamW8bitKahan"
elif optimizer == "AdamWScheduleFreePlus":
  optimizer = "LoraEasyCustomOptimizer.adamw_schedulefree_plus.AdamWScheduleFreePlus"

lr_scheduler_num_cycles = lr_scheduler_number
lr_scheduler_power = lr_scheduler_number

# Misc
import random
#seed = random.randint(0, 2**32)
seed = 1557
bucket_reso_steps = 64
min_bucket_reso = 256
max_bucket_reso = 4096


root_dir = "/content"
trainer_dir = os.path.join(root_dir, "trainer")
kohya_dir = os.path.join(trainer_dir, "sd_scripts")

venv_python = os.path.join(kohya_dir, "venv/bin/python")
venv_pip = os.path.join(kohya_dir, "venv/bin/pip")
train_network = os.path.join(kohya_dir, "anima_train_network.py" if is_anima else "sdxl_train_network.py")

if "/Loras" in folder_structure:
    main_dir = os.path.join(root_dir, "drive/MyDrive/Loras")
    log_folder = os.path.join(main_dir, "_logs")
    config_folder = os.path.join(main_dir, project_name)
    images_folder = os.path.join(main_dir, project_name, "dataset")
    output_folder = os.path.join(main_dir, project_name, "output")
else:
    main_dir = os.path.join(root_dir, "drive/MyDrive/lora_training")
    images_folder = os.path.join(main_dir, "datasets", project_name)
    output_folder = os.path.join(main_dir, "output", project_name)
    config_folder = os.path.join(main_dir, "config", project_name)
    log_folder = os.path.join(main_dir, "log")

config_file = os.path.join(config_folder, "training_config.toml")
dataset_config_file = os.path.join(config_folder, "dataset_config.toml")

def apply_patches():
    global kohya_dir, venv_pip, wandb_key

    print("\n🔧 Applying patches to the trainer...")
    os.chdir(kohya_dir)

    # 1. 이미지 로딩 및 에폭 이름 관련 패치
    if LOAD_TRUNCATED_IMAGES:
        !sed -i 's/from PIL import Image/from PIL import Image, ImageFile\nImageFile.LOAD_TRUNCATED_IMAGES=True/g' library/train_util.py
    if BETTER_EPOCH_NAMES:
        !sed -i 's/{:06d}/{:02d}/g' library/train_util.py
        if os.path.exists("train_network.py"):
            !sed -i 's/"." + args.save_model_as)/"-{:02d}.".format(num_train_epochs) + args.save_model_as)/g' train_network.py

    # 2. Diffusers 버전 경고 수정
    if FIX_DIFFUSERS:
        deprecation_utils = os.path.join(kohya_dir, "venv/lib/python3.11/site-packages/diffusers/utils/deprecation_utils.py")
        if os.path.exists(deprecation_utils):
            !sed -i 's/if version.parse/if False:#/g' {deprecation_utils}

    print("✅ All patches applied successfully.")

def install_trainer():
    global main_dir
    cache_filename = "kohya_trainer_cache_0527.tar.gz"
    cache_path = os.path.join(main_dir, cache_filename)
    # -------------------------------------------------------------------------

    print("\n📦 Installing system dependencies...")
    !apt -y update -qq
    !apt install -y python3.11-venv python3.11-dev aria2 pigz -qq

    # 1. 캐시가 있으면 복원
    if cache_installation and os.path.exists(cache_path):
        print(f"\n🚀 Cache found at {cache_path}!")
        print("⏳ Restoring installation with multi-core processing (pigz)...")
        t_start = time()
        !tar -I pigz -xf "{cache_path}" -C /content
        t_end = time()
        print(f"✅ Restore complete in {int(t_end - t_start)} seconds.")

    else:
        # 2. 캐시가 없으면 처음부터 설치
        print("\n🛠️ No cache found. Installing from scratch...")
        !git clone https://github.com/67372a/LoRA_Easy_Training_scripts_Backend {trainer_dir}
        os.chdir(trainer_dir)
        !git reset --hard {COMMIT}

        !git submodule update --init --recursive

        !chmod 755 ./colab_install.sh
        !./colab_install.sh

        # ProdigyPlusScheduleFree 설치
        print("⬇️ Installing latest ProdigyPlusScheduleFree...")
        !{venv_pip} install git+https://github.com/loganbooker/prodigy-plus-schedule-free.git

        os.chdir(trainer_dir)

        # fix logging
        !{venv_pip} uninstall -y rich

        !{venv_pip} install matplotlib-inline

        apply_patches()

        # 3. 캐시 저장
        if cache_installation:
            print(f"\n💾 Saving updated installation to cache (using pigz): {cache_path}...")
            !tar -I pigz -cf "{cache_path}" -C /content trainer
            print("✅ Cache saved with updated requirements.")

    # 환경변수 설정
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
    os.environ["BITSANDBYTES_NOWELCOME"] = "1"
    os.environ["SAFETENSORS_FAST_GPU"] = "1"
    os.environ["PYTHONWARNINGS"] = "ignore"
    os.chdir(root_dir)

def validate_dataset():
    global lr_warmup_steps, lr_warmup_ratio, caption_extension, keep_tokens, model_url
    supported_types = (".png", ".jpg", ".jpeg", ".webp", ".bmp")

    if model_url and model_url.startswith("/content/drive/") and not os.path.exists(model_url):
        print("💥 Error: The custom training model you specified was not found in your Google Drive.")
        return

    print("\n💿 Checking dataset...")
    if not project_name.strip() or any(c in project_name for c in " .()\"'\\/"):
        print("💥 Error: Please choose a valid project name.")
        return

    # Find the folders and files
    if custom_dataset:
        try:
            datconf = toml.loads(custom_dataset)
            datasets = [d for d in datconf["datasets"][0]["subsets"]]
        except:
            print(f"💥 Error: Your custom dataset is invalid or contains an error! Please check the original template.")
            return
        reg = [d.get("image_dir") for d in datasets if d.get("is_reg", False)]
        datasets_dict = {d["image_dir"]: d["num_repeats"] for d in datasets}
        folders = datasets_dict.keys()
        files = [f for folder in folders for f in os.listdir(folder)]
        images_repeats = {folder: (len([f for f in os.listdir(folder) if f.lower().endswith(supported_types)]), datasets_dict[folder]) for folder in folders}
    else:
        reg = []
        folders = []
        files = []
        images_repeats = {}

        # 최상위 폴더의 파일들 확인 (기본 설정된 num_repeats 사용)
        top_files = [f for f in os.listdir(images_folder) if os.path.isfile(os.path.join(images_folder, f))]
        top_images = [f for f in top_files if f.lower().endswith(supported_types)]
        if top_images:
            folders.append(images_folder)
            files.extend(top_files)
            images_repeats[images_folder] = (len(top_images), num_repeats)

        # 하위 폴더 확인 및 폴더별 repeat 인식
        for item in os.listdir(images_folder):
            item_path = os.path.join(images_folder, item)
            if os.path.isdir(item_path):
                folders.append(item_path)
                sub_files = os.listdir(item_path)
                sub_images = [f for f in sub_files if f.lower().endswith(supported_types)]

                # 정규식을 통한 반복 횟수 추출
                folder_repeats = num_repeats
                match = re.match(r"^(\d+)_", item)
                if match:
                    folder_repeats = int(match.group(1))

                if sub_images:
                    files.extend(sub_files)
                    images_repeats[item_path] = (len(sub_images), folder_repeats)

    # Validation
    for folder in folders:
        if not os.path.exists(folder):
            print(f"💥 Error: The folder {folder.replace('/content/drive/', '')} doesn't exist.")
            return
    for folder, (img, rep) in images_repeats.items():
        if not img:
            print(f"💥 Error: Your {folder.replace('/content/drive/', '')} folder is empty.")
            return
    test_files = []
    for f in files:
        if f.startswith("."):
            continue

        if not f.lower().endswith((caption_extension, ".npz")) and not f.lower().endswith(supported_types):
            print(f"💥 Error: Invalid file in dataset: \"{f}\". Aborting.")
            return
        for ff in test_files:
            if f.endswith(supported_types) and ff.endswith(supported_types) \
                and os.path.splitext(f)[0] == os.path.splitext(ff)[0]:
                print(f"💥 Error: The files {f} and {ff} cannot have the same name. Aborting.")
                return
        test_files.append(f)

    if caption_extension and not [txt for txt in files if txt.lower().endswith(caption_extension)]:
        caption_extension = ""
    if continue_from_lora and not (continue_from_lora.endswith(".safetensors") and os.path.exists(continue_from_lora)):
        print(f"💥 Error: Invalid path to existing Lora. Example: /content/drive/MyDrive/Loras/example.safetensors")
        return

    pre_steps_per_epoch = sum(img*rep for (img, rep) in images_repeats.values())
    steps_per_epoch = pre_steps_per_epoch/train_batch_size
    actual_total_steps = max_train_steps or int((max_train_epochs * steps_per_epoch) / gradient_accumulation_steps)
    lr_warmup_steps = int(actual_total_steps * lr_warmup_ratio)

    for folder, (img, rep) in images_repeats.items():
        print("📁"+folder.replace("/content/drive/", "") + (" (Regularization)" if folder in reg else ""))
        print(f"📈 Found {img} images with {rep} repeats.")

    return True

def create_config():
    global dataset_config_file, config_file, model_file, vae_file, qwen3_file, is_anima

    current_opt_args = optimizer_args.copy() if optimizer_args else []
    if "AdamWScheduleFreePlus" in optimizer:
        current_opt_args.append(f"warmup_steps={lr_warmup_steps}")

    if override_config_file:
        config_file = override_config_file
        print(f"\n⭕ Using custom config file {config_file}")
    else:
        config_dict = {
            "network_arguments": {
                "unet_lr": unet_lr,
                "text_encoder_lr": text_encoder_lr if not cache_text_encoder_outputs else 0,
                "llm_adapter_lr": 0,
                "network_dim": network_dim,
                "network_alpha": network_alpha,
                "network_module": network_module,
                "network_args": network_args,
                "network_train_unet_only": text_encoder_lr == 0 or cache_text_encoder_outputs,
                "network_weights": continue_from_lora or None
            },
            "optimizer_arguments": {
                "learning_rate": unet_lr,
                "lr_scheduler": lr_scheduler if optimizer != "ProdigyPlusScheduleFree" else "constant",
                "lr_scheduler_num_cycles": lr_scheduler_num_cycles if lr_scheduler == "cosine_with_restarts" and optimizer != "ProdigyPlusScheduleFree" else None,
                "lr_scheduler_power": lr_scheduler_power if lr_scheduler == "polynomial" and optimizer != "ProdigyPlusScheduleFree" else None,
                "lr_warmup_steps": lr_warmup_steps if lr_scheduler not in ("cosine", "constant") and optimizer != "ProdigyPlusScheduleFree" else 0,
                "optimizer_type": "prodigyplus.ProdigyPlusScheduleFree" if optimizer == "ProdigyPlusScheduleFree" else ("prodigyopt.Prodigy" if optimizer == "Prodigy" else optimizer),
                "optimizer_args": current_opt_args or None,
                "loss_type": "l2",
                "max_grad_norm": 0.0 if (optimizer in ("ProdigyPlusScheduleFree", "AdamWScheduleFreePlus") and prodigy_disable_grad_clip) else 1.0,
            },
            "training_arguments": {
                "lowram": lowram,
                # "highvram": True,
                "pretrained_model_name_or_path": model_file,
                "qwen3": qwen3_file if is_anima else None,
                "timestep_sampling": "shift" if (is_anima and discrete_flow_shift_enabled) else ("sigmoid" if is_anima else None),
                "sigmoid_scale": sigmoid_scale if is_anima else None,
                "discrete_flow_shift": discrete_flow_shift if (is_anima and discrete_flow_shift_enabled) else None,
                "train_llm_adapter": False,
                "vae": vae_file,
                "max_train_steps": max_train_steps,
                "max_train_epochs": max_train_epochs,
                "vae_batch_size": 2 if lowram else 4,
                "train_batch_size": train_batch_size,
                "seed": seed,
                "max_token_length": 225,
                "t5_max_token_length": 512,
                "qwen3_max_token_length": 512,
                "sdpa": attention_mode == "torch",
                "attn_mode": attention_mode,
                # "debiased_estimation_loss": True,
                "min_snr_gamma": min_snr_gamma if min_snr_gamma_enabled else None,
                "ip_noise_gamma": ip_noise_gamma if ip_noise_gamma_enabled else None,
                "noise_offset": noise_offset_value if noise_offset_enabled else None,
                "no_half_vae": True,
                "gradient_checkpointing": gradient_checkpointing,
                "unsloth_offload_checkpointing": unsloth_offload_checkpointing,
                "torch_compile": torch_compile,
                "dynamo_backend": dynamo_backend if torch_compile else None,
                "gradient_accumulation_steps": gradient_accumulation_steps,
                "max_data_loader_n_workers": 2 if lowram else 8,
                "persistent_data_loader_workers": True,
                "mixed_precision": mixed_precision,
                "full_fp16": mixed_precision == "fp16" and full_precision,
                "full_bf16": mixed_precision == "bf16" and full_precision,
                "cache_latents": cache_latents,
                "cache_latents_to_disk": cache_latents_to_drive,
                "cache_text_encoder_outputs": cache_text_encoder_outputs,
                "min_timestep": 0,
                "max_timestep": 1000,
                "prior_loss_weight": 1.0,
                "multires_noise_iterations": 6 if multinoise else None,
                "multires_noise_discount": 0.3 if multinoise else None,
                "v_parameterization": vpred or None,
                "scale_v_pred_loss_like_noise_pred": vpred or None,
                "zero_terminal_snr": vpred or None,
                "flow_model": is_rectified_flow or None,
                "flow_use_ot": flow_use_ot,
                "contrastive_flow_matching": contrastive_flow_matching,
                "flow_timestep_distribution": "uniform" if (is_rectified_flow and flow_shift_enabled) else None,
                "flow_uniform_static_ratio": flow_shift_value if (is_rectified_flow and flow_shift_enabled) else None,
                "focal_frequency_loss": focal_frequency_loss if focal_frequency_loss else None,
                "focal_frequency_loss_weight": focal_frequency_loss_weight if focal_frequency_loss else None,
                "focal_frequency_loss_alpha": focal_frequency_loss_alpha if focal_frequency_loss else None,
                "wavelet_masking": wavelet_masking if wavelet_masking else None,
                "wavelet_mask_l_bound": wavelet_mask_l_bound if wavelet_masking else None,
                },
            "dataset_arguments": {
                "caption_dropout_rate": caption_dropout_rate,
                "caption_tag_dropout_rate": None if cache_text_encoder_outputs else caption_tag_dropout_rate,
                "caption_dropout_every_n_epochs": caption_dropout_every_n_epochs,
            },
            "saving_arguments": {
                "save_precision": save_precision,
                "save_model_as": "safetensors",
                "save_every_n_epochs": save_every_n_epochs,
                "save_last_n_epochs": keep_only_last_n_epochs,
                "output_name": project_name,
                "output_dir": output_folder,
                "log_prefix": project_name,
                "logging_dir": log_folder,
                "wandb_api_key": wandb_key or None,
                "log_with": "wandb" if wandb_key else None,
                "wandb_run_name": project_name,
            }
        }

        for key in config_dict:
            if isinstance(config_dict[key], dict):
                config_dict[key] = {k: v for k, v in config_dict[key].items() if v is not None}

        with open(config_file, "w") as f:
            f.write(toml.dumps(config_dict))
        print(f"\n📄 Config saved to {config_file}")

    if override_dataset_config_file:
        dataset_config_file = override_dataset_config_file
        print(f"⭕ Using custom dataset config file {dataset_config_file}")
    else:
        # 동적 서브셋 구성
        if custom_dataset:
            datasets_value = toml.loads(custom_dataset)["datasets"]
        else:
            subsets_list = []

            # 1. 최상위 폴더 검사
            top_images = [f for f in os.listdir(images_folder) if os.path.isfile(os.path.join(images_folder, f)) and f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp'))]
            if top_images:
                subsets_list.append({
                    "num_repeats": num_repeats,
                    "image_dir": images_folder,
                    "class_tokens": None if caption_extension else project_name
                })

            # 2. 하위 폴더 검사
            for item in os.listdir(images_folder):
                item_path = os.path.join(images_folder, item)
                if os.path.isdir(item_path):
                    sub_images = [f for f in os.listdir(item_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp'))]
                    if sub_images:
                        folder_repeats = num_repeats
                        match = re.match(r"^(\d+)_", item)
                        if match:
                            folder_repeats = int(match.group(1))

                        subsets_list.append({
                            "num_repeats": folder_repeats,
                            "image_dir": item_path,
                            "class_tokens": None if caption_extension else project_name
                        })

            datasets_value = [{"subsets": subsets_list}]

        dataset_config_dict = {
            "general": {
                "resolution": resolution,
                "shuffle_caption": shuffle_caption and not cache_text_encoder_outputs,
                "keep_tokens": keep_tokens,
                "flip_aug": flip_aug,
                "caption_extension": caption_extension,
                "enable_bucket": True,
                "bucket_no_upscale": False,
                "bucket_reso_steps": bucket_reso_steps,
                "min_bucket_reso": min_bucket_reso,
                "max_bucket_reso": max_bucket_reso,
                "caption_dropout_rate": caption_dropout_rate,
                "caption_tag_dropout_rate": None if cache_text_encoder_outputs else caption_tag_dropout_rate,
                "caption_dropout_every_n_epochs": caption_dropout_every_n_epochs,
                # "resize_interpolation": "nearest",
            },
            "datasets": datasets_value
        }

        for key in dataset_config_dict:
            if isinstance(dataset_config_dict[key], dict):
                dataset_config_dict[key] = {k: v for k, v in dataset_config_dict[key].items() if v is not None}

        with open(dataset_config_file, "w") as f:
            f.write(toml.dumps(dataset_config_dict))
        print(f"📄 Dataset config saved to {dataset_config_file}")

def download_model():
    global old_model_url, model_url, model_file, vae_file, qwen3_file

    if is_anima:
        print("🚀 Downloading Anima components (VAE, Qwen3) and Base Model using aria2c...")
        # 다운로드를 위한 폴더들을 미리 생성합니다.
        os.makedirs("/content/models/qwen3/split_files/text_encoders", exist_ok=True)
        os.makedirs("/content/models/vae/split_files/vae", exist_ok=True)
        os.makedirs("/content/models/anima/split_files/diffusion_models", exist_ok=True)

        vae_model = "/content/models/vae/split_files/vae/qwen_image_vae.safetensors"
        qwen3_model = "/content/models/qwen3/split_files/text_encoders/qwen_3_06b_base.safetensors"

        vae_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors"
        qwen3_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors"

        # aria2c를 사용해 빠르고 안정적으로 다운로드하는 내부 헬퍼 함수
        def download_with_aria2(url, dest_path):
            if not os.path.exists(dest_path):
                aria2_cmd = [
                    "aria2c", "--console-log-level=warn", "-c", "-s", "16", "-x", "16", "-k", "10M",
                    "-d", os.path.dirname(dest_path), "-o", os.path.basename(dest_path)
                ]
                if HF_TOKEN:
                    aria2_cmd.extend(["--header", f"Authorization: Bearer {HF_TOKEN}"])
                aria2_cmd.append(url)

                try:
                    subprocess.run(aria2_cmd, check=True)
                except subprocess.CalledProcessError as e:
                    print(f"\n💥 Error: aria2c download failed for {os.path.basename(dest_path)}. ({e})")
                    raise SystemExit("다운로드에 실패하여 프로세스를 종료합니다.")

        # 1. VAE 다운로드
        print("⬇️ Downloading VAE...")
        download_with_aria2(vae_url, vae_model)

        # 2. Qwen3(Text Encoder) 다운로드
        print("⬇️ Downloading Qwen3 Text Encoder...")
        download_with_aria2(qwen3_url, qwen3_model)

        vae_file = vae_model
        qwen3_file = qwen3_model

        # 커스텀 링크를 넣지 않은 경우에만 베이스 DiT 모델을 여기서 다운로드하고 함수를 종료합니다.
        if not optional_custom_training_model:
            print("🚀 Downloading base Anima DiT model...")
            if "v1.0" in training_model:
                anima_model = "/content/models/anima/split_files/diffusion_models/anima-base-v1.0.safetensors"
                anima_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-base-v1.0.safetensors"
            elif "Preview 3" in training_model:
                anima_model = "/content/models/anima/split_files/diffusion_models/anima-preview3-base.safetensors"
                anima_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-preview3-base.safetensors"
            elif "Preview 2" in training_model:
                anima_model = "/content/models/anima/split_files/diffusion_models/anima-preview2.safetensors"
                anima_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-preview2.safetensors"
            else:
                anima_model = "/content/models/anima/split_files/diffusion_models/anima-preview.safetensors"
                anima_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-preview.safetensors"

            # 3. Base DiT 모델 다운로드
            download_with_aria2(anima_url, anima_model)

            model_file = anima_model
            print("✅ Anima 모델 및 컴포넌트 다운로드 완료.")
            return True

        # 커스텀 링크가 있는 경우, VAE/Qwen3만 받은 상태에서 아래의 일반 다운로드 로직(구글 드라이브, civitai 등)으로 넘어갑니다.
        print("💡 Custom Anima model URL detected. Proceeding to download the custom DiT model...")

    real_model_url = model_url  # There was a reason for having a separate variable...

    if real_model_url.startswith("/content/drive/"):
        # Local model, already checked to exist
        model_file = real_model_url
        print(f"📁 Using local model file: {model_file}")

        # Validation
        if model_file.lower().endswith(".safetensors"):
            from safetensors.torch import load_file as load_safetensors
            try:
                test = load_safetensors(model_file)
                del test
            except:
                return False
        elif model_file.lower().endswith(".ckpt"):
            from torch import load as load_ckpt
            try:
                test = load_ckpt(model_file)
                del test
            except:
                return False
        return True

    else:
        # Downloadable model
        if load_diffusers:
            if 'huggingface.co' in real_model_url:
                match = re.search(r'huggingface\.co/([^/]+)/([^/]+)', real_model_url)
                if match:
                    username = match.group(1)
                    model_name = match.group(2)
                    model_file = f"{username}/{model_name}"
                    from huggingface_hub import HfFileSystem
                    fs = HfFileSystem()
                    existing_folders = set(fs.ls(model_file, detail=False))
                    necessary_folders = [ "scheduler", "text_encoder", "text_encoder_2", "tokenizer", "tokenizer_2", "unet", "vae" ]
                    if all(f"{model_file}/{folder}" in existing_folders for folder in necessary_folders):
                        print("🍃 Diffusers model identified.")  # Will be handled by kohya
                        return True
            raise ValueError("💥 Failed to load Diffusers model. If this model is not Diffusers, have you tried turning it off at the top of the colab?")

        # Define local filename
        if not model_file or old_model_url and old_model_url != model_url:
            if real_model_url.lower().endswith((".ckpt", ".safetensors")):
                model_file = f"/content{real_model_url[real_model_url.rfind('/'):]}"
            else:
                model_file = "/content/downloaded_model.safetensors"
                if os.path.exists(model_file):
                    !rm "{model_file}"

        # HuggingFace
        if m := re.search(r"(?:https?://)?(?:www\.)?huggingface\.co/[^/]+/[^/]+/blob", real_model_url):
            real_model_url = real_model_url.replace("blob", "resolve")
        # Civitai
        elif m := re.search(r"(?:https?://)?(?:www\\.)?civitai\.com/models/([0-9]+)(/[A-Za-z0-9-_]+)?", real_model_url):
            if m.group(2):
                model_file = f"/content{m.group(2)}.safetensors"
            if m := re.search(r"modelVersionId=([0-9]+)", real_model_url):
                real_model_url = f"https://civitai.com/api/download/models/{m.group(1)}"
            else:
                raise ValueError("💥 optional_custom_training_model contains a civitai link, but the link doesn't include a modelVersionId. You can also right click the download button to copy the direct download link.")

        # Download checkpoint
        print(f"🚀 Downloading model to {model_file}...")
        aria2c_cmd_parts = [
            'aria2c', '--console-log-level=warn',
            '-c', '-s', '16', '-x', '16', '-k', '10M',
            '--user-agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            '-d', '/',
            '-o', model_file
        ]

        download_url_with_token = real_model_url
        if "civitai.com" in real_model_url and CIVITAI_TOKEN:
            print(" Civitai 토큰을 사용하여 다운로드합니다.")
            if '?' in download_url_with_token:
                download_url_with_token += f"&token={CIVITAI_TOKEN}"
            else:
                download_url_with_token += f"?token={CIVITAI_TOKEN}"

        elif "huggingface.co" in real_model_url and HF_TOKEN:
            print(" Hugging Face 토큰을 사용하여 다운로드합니다.")
            aria2c_cmd_parts.append('--header')
            aria2c_cmd_parts.append(f'Authorization: Bearer {HF_TOKEN}')

        aria2c_cmd_parts.append(download_url_with_token)

        # shell=False로 실행하여 명령어 인자를 리스트로 전달합니다.
        try:
          subprocess.run(aria2c_cmd_parts, check=True)
        except subprocess.CalledProcessError as e:
          print(f"\n💥 Error: aria2c download failed. Server might be down or token is invalid. ({e})")
          return False

        # Validation
        if model_file.lower().endswith(".safetensors"):
            from safetensors.torch import load_file as load_safetensors
            try:
                test = load_safetensors(model_file)
                del test
            except:
                new_model_file = os.path.splitext(model_file)[0]+".ckpt"
                !mv "{model_file}" "{new_model_file}"
                model_file = new_model_file
                print(f"Renamed model to {os.path.splitext(model_file)[0]}.ckpt")

        if model_file.lower().endswith(".ckpt"):
            from torch import load as load_ckpt
            try:
                test = load_ckpt(model_file)
                del test
            except:
                return False

        return True

def main():
    global dependencies_installed, images_folder, precision, mixed_precision, full_precision, optimizer, optimizer_args, max_train_steps, max_train_epochs, save_every_n_epochs

    if not os.path.exists('/content/drive'):
        from google.colab import drive
        print("📂 Connecting to Google Drive...")
        drive.mount('/content/drive')

    # 1. 훈련에 필요한 기본 디렉토리들을 먼저 생성합니다.
    for dir in (main_dir, trainer_dir, log_folder, output_folder, config_folder):
        os.makedirs(dir, exist_ok=True)

    # 2. 태거(Tagger) 스크립트를 사용하기 위해 가장 먼저 Kohya-ss 백엔드와 환경을 설치합니다.
    if not dependencies_installed:
        print("\n🏭 Installing trainer...\n")
        t0 = time()
        install_trainer()
        t1 = time()
        dependencies_installed = True
        print(f"\n✅ Installation finished in {int(t1-t0)} seconds.")
    else:
        print("\n✅ Dependencies already installed.")

    # 3. 원본 경로(구글 드라이브)를 로컬 임시 경로(/content/dataset)로 교체하고 압축을 해제합니다.
    images_folder = prepare_dataset_locally(images_folder, "/content/dataset", project_name)

    # 4. 자동 태깅을 실행합니다.
    auto_wd14_tagging(images_folder)

    # 5. 텍스트 파일(.txt)이 모두 생성된 이후에 유효성 검사를 진행해야 정상적으로 인식됩니다.
    if not validate_dataset():
        return

    # 6. 모델 다운로드 진행
    if old_model_url != model_url or not model_file or (not load_diffusers and not os.path.exists(model_file)):
        print("\n🔄 Getting model...")
        if not download_model():
            print("\n💥 Error: The model you specified is invalid or corrupted."
                  "\nIf you're using an URL, please check that the model is accessible without being logged in."
                  "\nYou can try civitai or huggingface URLs, or a path in your Google Drive starting with /content/drive/MyDrive")
            return
        print()
    else:
        print("\n🔄 Model already downloaded.\n")

    create_config()

    #-------------------------------------------------------------------------------------------

    !find /content/trainer/sd_scripts/library -name "*.py" -type f -exec sed -i '/epoch is incremented/d' {} +

    # # PyTorch 2.10 아키텍처와 충돌을 일으키는 torchao 패키지를 훈련 직전에 강제로 제거합니다.
    # !{venv_pip} uninstall -y torchao

    # !{venv_pip} install "numpy<2" "matplotlib==3.7.5" "contourpy==1.2.1" --upgrade -q
    # !/content/trainer/sd_scripts/venv/bin/pip install "opencv-python-headless==4.9.0.80"

    #임시 SDXL 언패킹 오류 패치
    if os.path.exists(train_network):
        !sed -i 's/encoder_hidden_states1, encoder_hidden_states2, pool2 = text_conds/encoder_hidden_states1, encoder_hidden_states2, pool2, *_ = text_conds/g' {train_network}

    # -------------------------------------------------------------------------------------------
    # Anima 학습 시에만 LyCORIS norms.py bias=None 및 weight=None 오류 패치 적용
    if is_anima:
        norms_py_path = os.path.join(kohya_dir, "venv/lib/python3.11/site-packages/lycoris/modules/norms.py")
        if os.path.exists(norms_py_path):
            with open(norms_py_path, 'r', encoding='utf-8') as f:
                norms_code = f.read()

            # 패치가 중복 적용되지 않도록 확인
            if "getattr(org_module, \"weight\", None) is None:" not in norms_code:

                # 1. weight가 없는 순수 수학적 Norm 레이어(AdaLN 등)는 학습에서 제외 (NoneType 'to' 에러 방지)
                norms_code = norms_code.replace(
                    'if self.module_type == "unknown":',
                    'if getattr(org_module, "weight", None) is None:\n            self.not_supported = True\n            return\n        if self.module_type == "unknown":'
                )

                # 2. __init__ 수정: self.b_norm = None 기본값 추가 및 bias None 검사
                norms_code = norms_code.replace(
                    'self.w_norm = nn.Parameter(torch.zeros(self.dim))\n        if hasattr(org_module, "bias"):',
                    'self.w_norm = nn.Parameter(torch.zeros(self.dim))\n        self.b_norm = None\n        if hasattr(org_module, "bias") and org_module.bias is not None:'
                )

                # 3. make_weight 수정: bias None 검사 추가
                norms_code = norms_code.replace(
                    'if hasattr(self.org_module[0], "bias"):',
                    'if hasattr(self.org_module[0], "bias") and self.org_module[0].bias is not None:'
                )

                with open(norms_py_path, 'w', encoding='utf-8') as f:
                    f.write(norms_code)
                print("✅ LyCORIS norms.py weight/bias error patched successfully for Anima.")

    # LyCORIS 패키지가 설치된 경로 내의 gora_utils.py 파일 지정
    gora_utils_path = "/content/trainer/sd_scripts/venv/lib/python3.11/site-packages/lycoris/modules/gora_utils.py"

    if os.path.exists(gora_utils_path):
        with open(gora_utils_path, 'r', encoding='utf-8') as f:
            code = f.read()

        # 패치가 중복 적용되는 것을 방지하기 위한 안전 장치입니다.
        if "ref_rank = int(ref_rank)" not in code:

            # 1. gora_precompute_gradients 함수 진입부에 형변환 코드 삽입
            # 원래 코드의 주석 및 형태를 그대로 보존하기 위해 특정 주석 라인을 타겟으로 삼습니다.
            target_phase_1 = "    # --- Phase 1: Gradient Accumulation ---"

            # 멀티라인 스트링 대신 괄호를 이용한 문자열 결합을 사용하여 정확히 4칸의 공백만 주입합니다.
            patch_phase_1 = (
                "    # UI/CLI로부터 전달된 문자열 파라미터를 올바른 숫자형으로 강제 변환합니다.\n"
                "    ref_rank = int(ref_rank)\n"
                "    if min_rank is not None: min_rank = int(min_rank)\n"
                "    if max_rank is not None: max_rank = int(max_rank)\n"
                "    scaling_alpha = float(scaling_alpha)\n"
                "    stable_gamma = float(stable_gamma)\n"
                "    convergence_threshold = float(convergence_threshold)\n\n"
                "    # --- Phase 1: Gradient Accumulation ---"
            )

            code = code.replace(target_phase_1, patch_phase_1)

            # 2. allocate_ranks 함수 내부의 min_rank, max_rank 비교 구문 형변환
            target_allocate = "rank = min(max(allocate_func(rank), min_rank), max_rank)"
            patch_allocate = "rank = min(max(allocate_func(rank), int(min_rank)), int(max_rank))"

            code = code.replace(target_allocate, patch_allocate)

            # 수정된 내용을 파일에 덮어씁니다.
            with open(gora_utils_path, 'w', encoding='utf-8') as f:
                f.write(code)

            print("✅ GoRA 문자열-정수 타입 충돌 에러 패치가 성공적으로 적용되었습니다.")
        else:
            print("✅ 이미 GoRA 패치가 적용되어 있습니다.")
    else:
        print("⚠️ gora_utils.py 파일을 찾을 수 없습니다.")

    os.chdir(kohya_dir)
    if os.path.exists("anima_train_network.py"):
        with open("anima_train_network.py", "r", encoding="utf-8") as f:
            code = f.read()
        if "model_timesteps = timesteps / 1000.0" not in code:

          # timesteps 스케일링 변수 분리 및 anima() 모델 호출 인자 수정
          code = code.replace("timesteps = timesteps / 1000.0", "model_timesteps = timesteps / 1000.0")
          code = re.sub(r'(anima\(\s*[^,]+,\s*)timesteps,', r'\g<1>model_timesteps,', code)

          with open("anima_train_network.py", "w", encoding="utf-8") as f:
              f.write(code)

          print("✅ Anima LWD 패치가 성공적으로 적용되었습니다.")
        else:
          print("✅ 이미 LWD 패치가 적용되어 있습니다.")

    else:
        print("⚠️ anima_train_network.py 파일을 찾을 수 없습니다.")

    os.chdir(kohya_dir)
    print("\n🛡️ ScheduleFree 옵티마이저 NaN/Inf 방어 패치를 검사합니다...")
    target_files = ["train_network.py", "sdxl_train_network.py", "anima_train_network.py"]
    for target_file in target_files:
        target_path = os.path.join(kohya_dir, target_file)
        if os.path.exists(target_path):
            with open(target_path, "r", encoding="utf-8") as f:
                code = f.read()

            # 패치가 이미 적용되었는지 확인 후 진행
            if "is_finite = True" not in code and "_raw_optimizer.step_func" in code:
                # 정규식을 통해 기존 코드의 들여쓰기를 유지하며 방어 로직 삽입
                pattern = r'([ \t]+)(_raw_optimizer\.step_func\([^)]+\))'
                replacement = r'''\1is_finite = True
\1for param_group in _raw_optimizer.param_groups:
\1    for param in param_group['params']:
\1        if param.grad is not None and not torch.isfinite(param.grad).all():
\1            is_finite = False
\1            break
\1    if not is_finite:
\1        break
\1if is_finite:
\1    \2
\1else:
\1    accelerator.print("Gradient overflow detected. Skipping optimizer step for ScheduleFree.")'''

                code = re.sub(pattern, replacement, code)

                with open(target_path, "w", encoding="utf-8") as f:
                    f.write(code)
                print(f"✅ {target_file}: ScheduleFree 방어 패치 적용 완료.")
            elif "_raw_optimizer.step_func" in code:
                print(f"✅ {target_file}: 이미 ScheduleFree 방어 패치가 적용되어 있습니다.")

    print("\n⭐ Starting trainer...\n")

    import glob
    img_exts = ('*.png', '*.jpg', '*.jpeg', '*.webp', '*.bmp')
    img_count = sum(len(glob.glob(os.path.join(images_folder, '**', ext), recursive=True)) for ext in img_exts)
    npz_count = len(glob.glob(os.path.join(images_folder, '**', '*.npz'), recursive=True))

    is_already_cached = (img_count > 0 and npz_count >= img_count)

    # 조건: lowram, mixed fp16 사용 목적, 디스크 캐싱 활성화 (Anima 학습 시 제외)
    if not is_anima and lowram and mixed_precision == "fp16" and not full_precision and cache_latents_to_drive and not is_already_cached:
        print("T4 캐싱 OOM 방지를 위해 '캐싱'과 '학습'을 분리 실행합니다.")

        # --- [1단계 백업]: 사용자의 원래 설정값 기억 ---
        orig_prec = precision
        orig_mixed = mixed_precision
        orig_full = full_precision
        orig_opt = optimizer
        orig_opt_args = optimizer_args.copy() if optimizer_args else []

        # --- [1단계 실행]: 캐싱 전용 모드 (의도적 크래시 유발) ---
        precision = "full fp16"
        mixed_precision = "fp16"
        full_precision = True
        optimizer = "AdamW8bit"

        #AdamW8bit에 존재하지 않는 파라미터를 강제로 주입하여 초기화 단계에서 크래시 유도
        optimizer_args = ["crash_after_caching=True"]

        create_config() # 1단계용 임시 config.toml 생성

        print("\n⏳ 1/2단계: Full FP16 환경에서 VAE를 로드하여 안전하게 Latent 캐싱을 수행합니다...")

        os.chdir(kohya_dir)
        # 의도적으로 에러가 발생하지만, Colab 셀 특성상 다음 코드로 실행이 넘어갑니다.
        !{venv_python} {train_network} --config_file={config_file} --dataset_config={dataset_config_file}
        os.chdir(root_dir)

        # --- [2단계 복구]: 원래 설정으로 되돌리기 ---
        print("\n🚀 2/2단계: 캐싱 완료! 정밀도를 Mixed FP16으로 변경하고 실제 학습을 시작합니다...")

        precision = orig_prec
        mixed_precision = orig_mixed
        full_precision = orig_full
        optimizer = orig_opt
        optimizer_args = orig_opt_args

        create_config() # 2단계용 원본 config.toml 재생성

        os.chdir(kohya_dir)
        !{venv_python} {train_network} --config_file={config_file} --dataset_config={dataset_config_file}
        os.chdir(root_dir)

    else:
        # 캐싱이 이미 되어있거나, 2단계 실행 조건이 아닐 경우 바로 본 학습 실행
        if is_already_cached:
            print("✅ 이미 디스크에 캐싱된 데이터(.npz)를 발견했습니다. 즉시 본 학습을 시작합니다!")


        os.chdir(kohya_dir)
        !{venv_python} {train_network} --config_file={config_file} --dataset_config={dataset_config_file}
        os.chdir(root_dir)

    if not get_ipython().__dict__['user_ns']['_exit_code']:
        display(Markdown("### ✅ Done! [Go download your Lora from Google Drive](https://drive.google.com/drive/my-drive)\n"
                         "### There will be several files, you should try the latest version (the file with the largest number next to it)"))


def prepare_dataset_locally(drive_path, local_path, current_project_name):
    # 프로젝트 이름을 기록할 마커 파일 경로 설정
    marker_file = os.path.join(local_path, ".project_name")

    # 1. 기존 데이터셋 폴더가 존재하는 경우 검사
    if os.path.exists(local_path) and os.listdir(local_path):
        # 마커 파일이 존재하면 읽어서 현재 프로젝트 이름과 대조
        if os.path.exists(marker_file):
            with open(marker_file, 'r') as f:
                loaded_project = f.read().strip()

            # 이름이 같으면 기존 데이터 재사용 (시간 절약)
            if loaded_project == current_project_name:
                print(f"✅ Dataset for '{current_project_name}' already loaded at {local_path}")
                return local_path
            else:
                # 이름이 다르면 (프로젝트 변경됨) 기존 폴더 삭제
                print(f"🔄 Project name changed ({loaded_project} -> {current_project_name}). Clearing old dataset...")
                shutil.rmtree(local_path)
        else:
            # 마커 파일이 없는 알 수 없는 폴더인 경우 안전을 위해 삭제
            print(f"🧹 Clearing unknown dataset at {local_path}...")
            shutil.rmtree(local_path)

    # 2. 깨끗한 상태에서 폴더 다시 생성
    os.makedirs(local_path, exist_ok=True)
    print(f"📂 Preparing dataset for '{current_project_name}'...")

    # 드라이브 경로가 ZIP 파일인 경우 (.zip 자동 감지 포함)
    zip_path = drive_path if drive_path.endswith(".zip") else drive_path + ".zip"

    if not os.path.exists(drive_path) and os.path.exists(zip_path):
        drive_path = zip_path
        print(f"  👉 Found ZIP file: {drive_path}")

    # 실제 복사 및 압축 해제 수행
    if drive_path.endswith(".zip"):
        if not os.path.exists(drive_path):
             print(f"💥 Error: Zip file not found at {drive_path}")
             return drive_path

        temp_zip = os.path.join("/content", os.path.basename(drive_path))
        print(f"  Step 1: Copying zip to local disk...")
        subprocess.run(['cp', drive_path, temp_zip], check=True)

        print(f"  Step 2: Unzipping to {local_path}...")
        subprocess.run(['unzip', '-q', '-o', temp_zip, '-d', local_path], check=True)
        os.remove(temp_zip)

    else:
        if not os.path.exists(drive_path):
             print(f"💥 Error: Dataset folder not found at {drive_path}")
             return drive_path

        print(f"  Copying folder directly to {local_path}...")
        # 앞서 rmtree로 지웠지만, copytree는 대상 폴더가 아예 없어야 작동하므로 다시 지움
        if os.path.exists(local_path):
            shutil.rmtree(local_path)
        shutil.copytree(drive_path, local_path)

    # 3. 준비가 완료되면 현재 프로젝트 이름을 마커 파일로 저장
    with open(marker_file, 'w') as f:
        f.write(current_project_name)

    print("✅ Dataset ready on fast local disk!")
    return local_path

def auto_wd14_tagging(dataset_dir):
    import glob
    import subprocess
    import os
    global kohya_dir, venv_python, venv_pip, custom_trigger_word

    # 지원하는 이미지 확장자 스캔
    img_exts = ('*.png', '*.jpg', '*.jpeg', '*.webp', '*.bmp')
    img_files = []
    for ext in img_exts:
        img_files.extend(glob.glob(os.path.join(dataset_dir, '**', ext), recursive=True))

    txt_files = glob.glob(os.path.join(dataset_dir, '**', '*.txt'), recursive=True)

    # 이미지 개수보다 텍스트 파일 개수가 적다면 태깅이 누락된 것으로 간주합니다.
    if len(txt_files) < len(img_files):
        print(f"\n🔍 캡션(.txt) 파일이 부족합니다. (이미지: {len(img_files)}장 / 캡션: {len(txt_files)}개)")
        print("⏳ 누락된 이미지에 대해 WD14 자동 태깅을 시작합니다...")

        tagger_script = os.path.join(kohya_dir, "finetune", "tag_images_by_wd14_tagger.py")

        print("📦 설치 확인: onnx 및 필수 비전 패키지 설치 중...")
        subprocess.run([
            venv_pip, "install", "-q",
            "onnx", "onnxruntime-gpu", "huggingface-hub",
            "huggingface-hub[hf_transfer]", "timm", "opencv-python-headless"
        ], check=True)

        tag_cmd = [
            venv_python, tagger_script,
            "--batch_size", "4",
            "--general_threshold", "0.35",
            "--character_threshold", "0.5",
            "--repo_id", "SmilingWolf/wd-vit-tagger-v3",
            "--recursive",
            "--remove_underscore",
            "--onnx",
            "--force_download",
            "--max_data_loader_n_workers", "2",
            dataset_dir
        ]

        try:
            print("⚙️ 태깅 스크립트를 실행합니다...")
            subprocess.run(tag_cmd, check=True)
            print("✅ WD14 자동 태깅이 성공적으로 완료되었습니다!")

            if custom_trigger_word:
                print(f"✍️ 생성된 모든 캡션 파일의 맨 앞에 트리거 워드('{custom_trigger_word}')를 추가합니다...")

                new_txt_files = glob.glob(os.path.join(dataset_dir, '**', '*.txt'), recursive=True)
                for txt_path in new_txt_files:
                    with open(txt_path, 'r', encoding='utf-8') as f:
                        content = f.read().strip()

                    # 중복 방지: 텍스트 맨 앞이 트리거 워드로 시작하지 않는 경우에만 추가
                    if not content.startswith(custom_trigger_word):
                        new_content = f"{custom_trigger_word}, {content}"
                        with open(txt_path, 'w', encoding='utf-8') as f:
                            f.write(new_content)

                print("✅ 트리거 워드 프리픽스(Prefix) 작업이 완료되었습니다!")
            else:
                # UI 입력창이 비어있을 경우 작동합니다.
                print("ℹ️ 트리거 워드가 공백으로 설정되어 프리픽스 추가 작업을 생략합니다.")

        except subprocess.CalledProcessError as e:
            print(f"\n💥 태깅 과정 중 오류가 발생했습니다. (종료 코드 {e.returncode})")
    else:
        print(f"\n✅ 모든 이미지({len(img_files)}장)에 캡션 파일이 존재합니다. 자동 태깅을 건너뜁니다.")

main()

## *️⃣ Extras

You can run these before starting the training.

In [ ]:
import torch
print("PyTorch 버전:", torch.__version__)
print("CUDA 버전:", torch.version.cuda)

PyTorch 버전: 2.11.0+cu128
CUDA 버전: 12.8


### 📚 Multiple folders in dataset
Below is a template allowing you to define multiple folders in your dataset. You must include the location of each folder and you can set different number of repeats for each one. To add more folders simply copy and paste the sections starting with `[[datasets.subsets]]`.

When enabling this, the number of repeats set in the main cell will be ignored, and the main folder set by the project name will also be ignored.

You can make one of them a regularization folder by adding `is_reg = true`  
You can also set different `keep_tokens`, `flip_aug`, etc.

In [ ]:
custom_dataset = """
[[datasets]]

[[datasets.subsets]]
image_dir = "/content/drive/MyDrive/Loras/example/dataset/good_images"
num_repeats = 3

[[datasets.subsets]]
image_dir = "/content/drive/MyDrive/Loras/example/dataset/normal_images"
num_repeats = 1

"""

In [ ]:
custom_dataset = None

In [ ]:
#@markdown ### 📂 Unzip dataset
#@markdown It's much slower to upload individual files to your Drive, so you may want to upload a zip if you have your dataset in your computer.
zip_path = "/content/drive/MyDrive/lora_training/aaaa.zip" #@param {type:"string"}
extract_to = "/content/drive/MyDrive/lora_training/datasets/aaaa" #@param {type:"string"}

import os

# 구글 드라이브 마운트 확인
if not os.path.exists('/content/drive'):
  from google.colab import drive
  print("📂 Connecting to Google Drive...")
  drive.mount('/content/drive')

# 목표 폴더 생성
os.makedirs(extract_to, exist_ok=True)

# 7zip 설치 (코랩에 기본적으로 없거나 구버전일 수 있음)
print("📦 Installing 7zip...")
!apt-get update -qq
!apt-get install -y p7zip-full -qq

# 7zip으로 압축 해제 실행
# -y: 모든 질문에 Yes (덮어쓰기 등)
# -o: 출력 경로 지정 (붙여서 써야 함)
print(f"🚀 Extracting {zip_path} to {extract_to}...")
!7z x "{zip_path}" -o"{extract_to}" -y

print("✅ Done")

In [ ]:
#@markdown  ### 🔢 datasets 폴더 다운로드
#@markdown 학습한 project_name과 같게 설정하여 태깅 및 캐싱이 완료된 datasets 폴더를 구글드라이브로 다운로드합니다.
import os
import shutil

# 프로젝트 이름 설정 (맨 위 설정과 동일하게)
project_name = "" #@param {type:"string"}
# 저장 경로 설정 (본인 드라이브 구조에 맞게 수정 가능)
save_dir = "/content/drive/MyDrive/lora_training/datasets"

source_dir = "/content/dataset"
output_filename = f"{project_name}_cached"
output_path = os.path.join(save_dir, output_filename)

if os.path.exists(source_dir):
    print(f"📦 Zipping '{source_dir}' to '{output_path}.zip'...")
    shutil.make_archive(output_path, 'zip', source_dir)
    print("✅ 완료! Google Drive에 .zip 파일이 생성되었습니다.")
else:
    print("❌ /content/dataset 폴더가 없습니다. 학습 코드를 먼저 실행했는지 확인하세요.")

In [ ]:
#@markdown ### 🔢 Count datasets
#@markdown Google Drive makes it impossible to count the files in a folder, so this will show you the file counts in all folders and subfolders.
folder = "/content/drive/MyDrive/lora_training/datasets/aaaa" #@param {type:"string"}

import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    print("📂 Connecting to Google Drive...\n")
    drive.mount('/content/drive')

tree = {}
exclude = ("_logs", "/output")
for i, (root, dirs, files) in enumerate(os.walk(folder, topdown=True)):
  dirs[:] = [d for d in dirs if all(ex not in d for ex in exclude)]
  images = len([f for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))])
  captions = len([f for f in files if f.lower().endswith(".txt")])
  others = len(files) - images - captions
  path = root[folder.rfind("/")+1:]
  tree[path] = None if not images else f"{images:>4} images | {captions:>4} captions |"
  if tree[path] and others:
    tree[path] += f" {others:>4} other files"

pad = max(len(k) for k in tree)
print("\n".join(f"📁{k.ljust(pad)} | {v}" for k, v in tree.items() if v))


In [ ]:
#@markdown ### ↪️ Continue

#@markdown Here you can write a path in your Google Drive to load an existing Lora file to continue training on.<p>
#@markdown **Warning:** It's not the same as one long training session. The epochs start from scratch, and it may have worse results.
continue_from_lora = "/content/drive/MyDrive/lora_training/aaaa.safetensors" #@param {type:"string"}
if continue_from_lora and not continue_from_lora.startswith("/content/drive/MyDrive"):
  import os
  continue_from_lora = os.path.join("/content/drive/MyDrive", continue_from_lora)


In [ ]:
#@title ## 🔍 LWD 마스크 시각화 테스트 (Visualizer)
#@markdown 학습 데이터셋의 이미지를 지정한 장수만큼 순차적으로 불러와 웨이블릿 마스크를 시각화합니다.

import os
import sys
import subprocess

# 1. 스마트 의존성 설치 (이미 설치된 경우 스킵)
try:
    import kornia
    import pywt
    import pytorch_wavelets
    print("✅ 의존성 패키지가 이미 설치되어 있습니다. (설치 스킵)")
except ImportError:
    print("📦 누락된 의존성 패키지를 최초 1회 설치합니다. (약 10~20초 소요)...")
    subprocess.run(
        "pip install -q kornia PyWavelets pytorch-wavelets==1.3.0 git+https://github.com/67372a/RamTorch",
        shell=True, check=True
    )
    print("✅ 설치 완료!")

import torch
import numpy as np
from PIL import Image
import random
import glob
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

#@markdown **테스트 설정**
#@markdown 테스트할 이미지 파일 또는 폴더 경로를 입력하세요. 비워두면 현재 업로드한 datasets 폴더를 대상으로 삼습니다.
test_image_path = "" #@param {type:"string"}
#@markdown 확인할 이미지 장 수 (데이터셋에서 무작위로 뽑아 연속으로 보여줍니다)
num_images = 10 #@param {type:"slider", min:1, max:20, step:1}
#@markdown 최대 해상도 (원본 비율을 유지하며 긴 쪽을 이 크기에 맞춥니다)
max_resolution = 1024 #@param {type:"slider", min:512, max:2048, step:128}
#@markdown 최소 마스크 하한선 (0.0 ~ 1.0)
wavelet_mask_l_bound = 0.3 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown 현재 노이즈 단계 (0 ~ 1000)
timestep = 500 #@param {type:"slider", min:0, max:1000, step:50}

def get_closest_multiple_of_64(size):
    return int(round(size / 64.0) * 64)

def run_multi_lwd_visualizer():
    # 2. sd-scripts 경로 설정 및 라이브러리 임포트
    sd_scripts_dir = "/content/trainer/sd_scripts"
    if not os.path.exists(sd_scripts_dir):
        display(Markdown("### ❌ 오류: 트레이너 백엔드를 찾을 수 없습니다.\n먼저 메인 학습 셀을 1회 이상 실행하여 설치를 완료해주세요."))
        return

    if sd_scripts_dir not in sys.path:
        sys.path.insert(0, sd_scripts_dir)

    try:
        from library.train_util import setup_wavelet_dwt, compute_wavelet_attention_map, get_wavelet_mask
    except ImportError as e:
        display(Markdown(f"### ❌ 오류: 필요한 라이브러리를 불러올 수 없습니다. ({e})\n런타임을 재시작하거나 셀을 다시 실행해보세요."))
        return

    # 3. 개선된 이미지/폴더 선택 및 스캔 로직
    img_path = test_image_path.strip().to_string() if hasattr(test_image_path, 'to_string') else str(test_image_path).strip()
    image_pool = []
    img_exts = ('*.png', '*.jpg', '*.jpeg', '*.webp', '*.bmp')

    # [정밀 분기 처리]
    if img_path and os.path.isfile(img_path):
        # Case A: 특정 이미지 파일 경로를 직접 넣은 경우
        image_pool = [img_path]
        print(f"🎯 입력된 이미지 파일을 인식했습니다.")
    elif img_path and os.path.isdir(img_path):
        # Case B: 특정 폴더 경로(구글 드라이브 포함)를 직접 넣은 경우
        print(f"📂 입력된 커스텀 폴더 경로를 인식했습니다: {img_path}")
        for ext in img_exts:
            image_pool.extend(glob.glob(os.path.join(img_path, '**', ext), recursive=True))
    else:
        # Case C: 비워두었거나 입력한 경로가 올바르지 않은 경우 (기본 데이터셋 폴더 자동 추적)
        target_folder = globals().get('images_folder', '/content/dataset')
        if not os.path.exists(target_folder):
            target_folder = "/content/drive/MyDrive/lora_training/datasets/" + globals().get('project_name', '')

        if os.path.exists(target_folder):
            print(f"🔍 자동 감지된 데이터셋 폴더를 스캔합니다: {target_folder}")
            for ext in img_exts:
                image_pool.extend(glob.glob(os.path.join(target_folder, '**', ext), recursive=True))

    if not image_pool:
        display(Markdown(f"### ❌ 오류: 지정한 경로에서 이미지 파일을 찾을 수 없습니다.\n입력 경로를 다시 확인해주세요: `{img_path}`"))
        return

    # 출력할 최종 이미지 장수 결정
    display_count = min(num_images, len(image_pool))
    selected_images = random.sample(image_pool, display_count) if len(image_pool) > 1 else image_pool
    print(f"총 {len(image_pool)}장의 이미지 중 {display_count}장을 순차적으로 분석합니다...\n")

    # 4. 모델 및 텐서 초기화 (루프 밖에서 1번만 실행)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("⏳ Wavelet DWT 모듈 초기화 중...")
    dwt_module = setup_wavelet_dwt(device)

    # 5. 다중 이미지 순차 처리 루프
    for idx, current_img_path in enumerate(selected_images):
        print(f"▶ [{idx+1}/{display_count}] 처리 중: {os.path.basename(current_img_path)}")

        img = Image.open(current_img_path).convert("RGB")
        w, h = img.size
        scale = min(max_resolution / w, max_resolution / h)

        new_w = max(64, get_closest_multiple_of_64(w * scale))
        new_h = max(64, get_closest_multiple_of_64(h * scale))

        try:
            resample_filter = Image.Resampling.LANCZOS
        except AttributeError:
            resample_filter = Image.LANCZOS

        img = img.resize((new_w, new_h), resample_filter)

        img_array = np.array(img).astype(np.float32) / 255.0
        tensor = torch.from_numpy(img_array).permute(2, 0, 1).unsqueeze(0).to(device)

        # 마스크 계산
        with torch.no_grad():
            attn_map = compute_wavelet_attention_map(tensor, dwt_module)
            t_tensor = torch.tensor([timestep]).float().to(device)
            mask = get_wavelet_mask(attn_map, l=wavelet_mask_l_bound, T=1000, timesteps=t_tensor)

        # 이미지 변환
        attn_img_np = (attn_map[0].cpu().numpy() * 255).astype(np.uint8)
        attn_img = Image.fromarray(attn_img_np, mode="L").convert("RGB")

        mask_np = (mask[0, 0].cpu().numpy() * 255).astype(np.uint8)
        mask_img = Image.fromarray(mask_np, mode="L").convert("RGB")

        red_overlay = Image.new("RGB", img.size, (255, 0, 0))
        mask_for_composite = Image.fromarray(mask_np, mode="L")
        composite_img = Image.composite(red_overlay, img, mask_for_composite)
        blended_composite = Image.blend(img, composite_img, alpha=0.6)

        # Matplotlib 렌더링
        fig, axes = plt.subplots(1, 4, figsize=(24, 8), dpi=150)

        axes[0].imshow(img)
        axes[0].set_title(f"1. Original ({new_w}x{new_h})", fontsize=14, fontweight='bold')
        axes[0].axis("off")

        axes[1].imshow(attn_img)
        axes[1].set_title("2. Frequency Attention Map", fontsize=14, fontweight='bold')
        axes[1].axis("off")

        axes[2].imshow(mask_img)
        axes[2].set_title(f"3. LWD Mask (l={wavelet_mask_l_bound}, t={timestep})", fontsize=14, fontweight='bold')
        axes[2].axis("off")

        axes[3].imshow(blended_composite)
        axes[3].set_title("4. Mask Overlay", fontsize=14, fontweight='bold')
        axes[3].axis("off")

        plt.tight_layout()
        plt.show()

# 실행
run_multi_lwd_visualizer()

In [ ]:
#@title ## 🔍 Focal Frequency Loss (FFL) 시각화 시뮬레이터
#@markdown 원본 이미지와 디테일이 뭉개진(Blur) 가짜 생성 이미지를 비교하여, FFL이 주파수 스펙트럼에서 어떻게 오차를 찾아내고 집중(Focal) 벌점을 부여하는지 확인합니다.

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from IPython.display import display, Markdown
import random
import glob
import os

#@markdown **테스트 설정**
#@markdown 테스트할 이미지 파일 경로. 비워두면 데이터셋에서 무작위 선택됩니다.
test_image_path = "" #@param {type:"string"}
#@markdown 모델이 디테일을 얼마나 뭉갰는지 시뮬레이션합니다 (가짜 생성 이미지의 흐림 정도)
blur_radius = 1 #@param {type:"slider", min:0.5, max:5.0, step:0.5}
#@markdown 집중도 알파 (`focal_frequency_loss_alpha`) - 값이 클수록 틀린 주파수에 더 가혹하게 집착합니다.
ffl_alpha = 1 #@param {type:"slider", min:0.0, max:3.0, step:0.5}

def run_ffl_simulator():
    # 1. 이미지 로드 및 흑백 변환 (주파수 분석을 직관적으로 보기 위해 1채널 사용)
    img_path = test_image_path.strip()
    if not img_path or not os.path.exists(img_path):
        target_folder = globals().get('images_folder', '/content/dataset')
        if not os.path.exists(target_folder):
            target_folder = "/content/drive/MyDrive/lora_training/datasets/" + globals().get('project_name', '')

        image_pool = []
        if os.path.exists(target_folder):
            for ext in ('*.png', '*.jpg', '*.jpeg', '*.webp'):
                image_pool.extend(glob.glob(os.path.join(target_folder, '**', ext), recursive=True))
        if image_pool:
            img_path = random.choice(image_pool)
            print(f"🎲 무작위 이미지 선택됨: {os.path.basename(img_path)}\n")
        else:
            display(Markdown("### ❌ 오류: 이미지를 찾을 수 없습니다."))
            return

    # 원본 이미지 (Real)
    img_orig = Image.open(img_path).convert("L").resize((512, 512))
    # 모델이 아직 학습이 덜 되어 디테일을 뭉갠 상황 시뮬레이션 (Fake)
    img_fake = img_orig.filter(ImageFilter.GaussianBlur(radius=blur_radius))

    # 텐서 변환
    t_orig = torch.tensor(np.array(img_orig) / 255.0)
    t_fake = torch.tensor(np.array(img_fake) / 255.0)

    # 2. 2D 이산 푸리에 변환 (FFT) 수행하여 주파수 도메인으로 이동
    # fftshift를 통해 저주파(형태)는 중앙으로, 고주파(디테일)는 가장자리로 모읍니다.
    fft_orig = torch.fft.fftshift(torch.fft.fft2(t_orig))
    fft_fake = torch.fft.fftshift(torch.fft.fft2(t_fake))

    # 사람이 눈으로 볼 수 있게 진폭(Magnitude)에 로그 스케일 적용
    mag_orig = torch.log(torch.abs(fft_orig) + 1e-7).numpy()

    # 3. 주파수 오차 (Distance) 계산
    # 원본 주파수와 가짜 이미지 주파수 간의 유클리디안 거리(차이)
    diff = torch.abs(fft_orig - fft_fake)

    # 4. Focal Weight (FFL의 핵심!) 계산
    # 오차가 큰 주파수 대역에 alpha 제곱을 하여 가중치를 극대화합니다.
    diff_norm = (diff - diff.min()) / (diff.max() - diff.min() + 1e-7) # 0~1 정규화
    focal_weight = (diff_norm ** ffl_alpha).numpy()

    diff_vis = torch.log(diff + 1e-7).numpy()

    # 5. 시각화 렌더링
    fig, axes = plt.subplots(2, 3, figsize=(18, 12), dpi=120)

    # 상단: 픽셀 공간 (눈으로 보는 이미지)
    axes[0, 0].imshow(img_orig, cmap='gray')
    axes[0, 0].set_title("1. Original Image (Real)", fontsize=14, fontweight='bold')
    axes[0, 0].axis("off")

    axes[0, 1].imshow(img_fake, cmap='gray')
    axes[0, 1].set_title(f"2. Blurred Image (Fake) - Blur: {blur_radius}", fontsize=14, fontweight='bold')
    axes[0, 1].axis("off")

    # 원본과 가짜의 단순 픽셀 차이
    pixel_diff = np.abs(np.array(img_orig) - np.array(img_fake))
    axes[0, 2].imshow(pixel_diff, cmap='hot')
    axes[0, 2].set_title("3. Spatial Error (Pixel Difference)", fontsize=14, fontweight='bold')
    axes[0, 2].axis("off")

    # 하단: 주파수 공간 (FFL이 보는 세상)
    axes[1, 0].imshow(mag_orig, cmap='viridis')
    axes[1, 0].set_title("4. 2D FFT Spectrum (Original)", fontsize=14, fontweight='bold')
    axes[1, 0].axis("off")

    axes[1, 1].imshow(diff_vis, cmap='magma')
    axes[1, 1].set_title("5. Frequency Error (Spectrum Diff)", fontsize=14, fontweight='bold')
    axes[1, 1].axis("off")

    axes[1, 2].imshow(focal_weight, cmap='jet')
    axes[1, 2].set_title(f"6. FFL Weight Map (Alpha={ffl_alpha})", fontsize=14, fontweight='bold', color='red')
    axes[1, 2].axis("off")

    plt.tight_layout()
    plt.show()

    summary = f"""
💡 **해석 가이드 (주파수 도메인 읽는 법):**
* **4번 이미지 (2D FFT):** 중앙의 밝은 점은 '저주파(전체적인 명암, 형태)'이고, 가장자리로 퍼져나가는 입자들은 '고주파(머리카락, 질감, 엣지)'입니다.
* **5번 이미지 (Frequency Error):** AI가 이미지를 뭉갰을 때(Blur), 원본과 비교해서 **어떤 주파수 대역이 날아갔는지(손실되었는지)** 오차를 보여줍니다. 주로 가장자리(고주파) 오차가 큽니다.
* **6번 이미지 (FFL Weight Map):** <span style="color:red">**FFL의 최종 결과물입니다.**</span> 5번의 오차를 바탕으로, **"지금 네가 가장 못 맞추고 있는 주파수가 저기니까, 저 주파수를 복구하는 데에 학습력(가중치)을 {ffl_alpha}배로 쏟아부어라!"** 라고 모델을 때리는 벌점 지도입니다.
    * `ffl_alpha` 슬라이더를 **3.0**으로 올리면, 모델이 특정 질감을 살리는 데 극단적으로 집착하게 되는 현상을 눈으로 볼 수 있습니다.
"""
    display(Markdown(summary))

run_ffl_simulator()

# 📈 Plot training results
You can do this after running the trainer. You don't need this unless you know what you're doing.  
The first cell below may fail to load all your logs. Keep trying the second cell until all data has loaded.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir={log_folder}/

In [ ]:
from tensorboard import notebook
notebook.display(port=6006, height=800)